# Flagged or Fraud? Instructor solution

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## Load data

In [ ]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [ ]:
train.head()

In [ ]:
train.describe()

In [ ]:
train.isna().sum()

## Preprocessing

In [ ]:
for df in [train, test]:
    df["never_chargeback"] = (df["days_since_last_chargeback"] == -1).astype(int)
    df["days_since_last_chargeback"] = np.log1p(df["days_since_last_chargeback"].replace(-1, np.nan))
    df["amount_vs_usual"] = np.log(df["amount_inr"] / df["customer_avg_spend_90d"])
    df["amount_is_round"] = (df["amount_inr"] % 500 == 0).astype(int)
    df["hour_sin"] = np.sin(2 * np.pi * df["transaction_hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["transaction_hour"] / 24)
    for col in ["amount_inr", "customer_avg_spend_90d", "card_age_days", "distance_from_home_km", "txn_count_last_1h"]:
        df[col] = np.log1p(df[col])
    for col in train.columns[train.isna().any()]:
        df[col + "_missing"] = df[col].isna().astype(int)

## Features

In [ ]:
FEATURES = train.select_dtypes("number").columns.drop("is_fraud").tolist()
FEATURES = [col for col in FEATURES if col not in ["batch_number", "acquirer_code", "pos_software_version"]]

TEXT = ["merchant_category", "channel", "card_type", "city_tier"]
X = pd.get_dummies(train[FEATURES + TEXT], columns=TEXT, dtype=float)
X_test = pd.get_dummies(test[FEATURES + TEXT], columns=TEXT, dtype=float)
X_test = X_test.reindex(columns=X.columns, fill_value=0)
y = train["is_fraud"]

## Train / validation split

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

medians = X_train.median()
X_train = X_train.fillna(medians)
X_val = X_val.fillna(medians)
X_test = X_test.fillna(medians)
scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

## Model

In [ ]:
keras.utils.set_random_seed(0)

model = keras.Sequential([
    keras.layers.Input(shape=(X_train.shape[1],)),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

## Training

In [ ]:
reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5)
history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                    epochs=100, batch_size=32, callbacks=[reduce_lr], verbose=2)

## Loss and accuracy curves

In [ ]:
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="validation")
plt.title("Loss")
plt.xlabel("epoch")
plt.legend()
plt.show()

plt.plot(history.history["accuracy"], label="train")
plt.plot(history.history["val_accuracy"], label="validation")
plt.title("Accuracy")
plt.xlabel("epoch")
plt.legend()
plt.show()

## Submission

In [ ]:
pred = (model.predict(X_test) > 0.5).astype(int).ravel()
submission = pd.DataFrame({"transaction_id": test["transaction_id"], "is_fraud": pred})
submission.to_csv("submission.csv", index=False)